Example of Jupyter "magic command":

In [ ]:
%pwd

To install DuckDB Python module:

In [ ]:
# %pip install duckdb
# %pip install pandas

In [ ]:

import duckdb
import pandas as pd

### 1. Create a connection to the database

In [ ]:
conn = duckdb.connect("~/GitHub/gitRDS/EDS213/bren-eds213-data/database/database.duckdb")

In [ ]:
conn

### 2. Now let's query the databse using:
  a. `.sql()` method -- specific to DuckDB and analysis focused


In [ ]:
conn.sql("SELECT * FROM Site LIMIT 5")


Note: This is also a lazy evaluation like we were doing with `dbplyr`. At this point, the data has not been fully processed or brought into Python memory yet. It's a symbolic representation of a query. To see the actual data, you need to materialize the relation. You can do this in several ways: 

  > `.show()`: Prints a nice tabular representation (great for interactive exploration).
  > `.fetchall()`: Returns all results as a list of tuples (the traditional DB-API way).
  > `.fetchone()`: Returns the next single row as a tuple.
  >`.df()`: Converts the result into a Pandas DataFrame.Now we want results... 


In [ ]:
# You get a list of tuples (one per row)
conn.sql("SELECT * FROM Site LIMIT 5").fetchall()


In [ ]:
# You get a pandas dataframe
site_df = conn.sql("SELECT * FROM Site").df()
site_df.head()

In [ ]:
site_table = conn.execute("SELECT * FROM Site")

site_table

  b. `.execute()` method -- more ubiquitous to other database workflows where you create a cursor object that you use to iterate on the rows of a table


In [ ]:
cur = conn.cursor()

Cursors don't store anything, they just transfer queries to the database and get results back.

In [ ]:
cur.execute("SELECT * FROM Site LIMIT 5")

We still need to fectch the results as before:

In [ ]:
cur.fetchall()

Always get tuples, even if you only request one column

In [ ]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")

In [ ]:
cur.fetchall()

In [ ]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")
[t[0] for t in cur.fetchall()]

3. Get the one result

In [ ]:
cur.execute("SELECT COUNT(*) FROM Bird_nests")
cur.fetchall()

In [ ]:
cur.execute("SELECT COUNT(*) FROM Bird_nests")
cur.fetchone()

In [ ]:
cur.execute("SELECT COUNT(*) FROM Bird_nests")
cur.fetchone()[0]

3. Using an iterator - but DuckDB doesn't support iterators :(

In [ ]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")
for row in cur:
    print(f"got {row[0]}")

A workaround is to use the `.fectchone()` method:

In [ ]:
cur.execute("SELECT Nest_ID FROM Bird_nests LIMIT 10")
while True:
    row = cur.fetchone()
    if row == None:
        break
    # do something with row
    print(f"got nest ID {row[0]}")

Can do things other than SELECT!

In [ ]:
cur.execute("CREATE TEMP TABLE temp_table AS
            SELECT * FROM Bird_nests LIMIT 10")

**Watch out for line breaks!!**

In [ ]:
cur.execute("""
    CREATE TEMP TABLE temp_table AS
    SELECT * FROM Bird_nests LIMIT 10
    """)

In [ ]:
cur.execute("SELECT * FROM temp_table")

In [ ]:
cur.fetchone()

### A note on fragility

For example:
```sql
INSERT INTO Site VALUES ("abcd", "Foo", 35.7, 42.3, "?")
```

A less fragile way of expressing the same thing:

```sql
INSERT INTO Site (Code, Site_name, Latitude, Longitude, Something_else)
   VALUES ("abcd", "Foo", 35.7, 42.3, "?")
```
   
In the same vein: `SELECT *` is fragile

In [ ]:
cur.execute("SELECT * FROM Site LIMIT 3")
cur.fetchall()

A better, more robust way of coding the same thing:

In [ ]:
cur.execute("SELECT Site_name, Code, Latitude, Longitude FROM Site LIMIT 3")
cur.fetchall()

An extended example: Question we're trying to answer: How many nests do we have for each species?

Approach: first get all species.  Then execute a count query for each species.

A digression: string interpolation in Python

In [ ]:
# The % method
s = "My name is %s"
print(s % "Greg")
s = "My name is %s and the other teacher's name is %s"
print(s % ("Greg", "Julien"))
# The new f-string method
name = "Greg"
print(f"My name is {name}")
# Third way
print("My name is {}".format("Greg"))

In [ ]:
query = """
   SELECT COUNT(*) FROM Bird_nests
   WHERE Species = '%s'
"""
cur.execute("SELECT Code FROM Species LIMIT 3")
for row in cur.fetchall():  # DuckDB workaround
    code = row[0]
    prepared_query = query % code
    #print(prepared_query)
    cur2 = conn.cursor()
    cur2.execute(prepared_query)
    print(f"Species {code} has {cur2.fetchone()[0]} nests")
    cur2.close()

The above Python interpolation is dangerous and has caused many database hacks!  There's a better way

In [ ]:
query = """
   SELECT COUNT(*) FROM Bird_nests
   WHERE Species = ?
"""
cur.execute("SELECT Code FROM Species LIMIT 3")
for row in cur.fetchall():  # DuckDB workaround
    code = row[0]
    # NOT NEEDED! prepared_query = query % code
    #print(prepared_query)
    cur2 = conn.cursor()
    cur2.execute(query, [code])  # <-- added argument here
    print(f"Species {code} has {cur2.fetchone()[0]} nests")
    cur2.close()

Let's illustrate the danger with a different example

In [ ]:
abbrev = "TS"
name = "Taylor Swift"
cur.execute("""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES ('%s', '%s')
   """ % (abbrev, name)
           )

In [ ]:
cur.execute("SELECT * FROM Personnel")
cur.fetchall()[-3:]

In [ ]:
abbrev = "CO"
name = "Conan O'Brien"
cur.execute("""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES ('%s', '%s')
   """ % (abbrev, name)
           )

In [ ]:
"""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES ('%s', '%s')
   """ % (abbrev, name)

In [ ]:
abbrev = "CO"
name = "Conan O'Brien"
cur.execute("""
   INSERT INTO Personnel (Abbreviation, Name)
   VALUES (?, ?)
   """,
    [abbrev, name])

In [ ]:
cur.execute("SELECT * FROM Personnel")
cur.fetchall()[-3:]

Don't forget to close your connection when you are done!

In [ ]:
conn.close()